# Video Language Detection

This notebook filters videos in Portuguese using automatic language detection based on a transformer model.

**Pipeline**: Concatenation (title + description) → Language detection (XLM-RoBERTa) → Filtering Portuguese videos

**Output**: A list of video IDs classified as Portuguese for downstream analysis

In [ ]:
import pandas as pd
import numpy as np
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch
from tqdm import tqdm
import os

In [ ]:
# Centralized paths
PROJECT_ROOT = os.getcwd()
CLEANED_DATA_DIR = os.path.join(PROJECT_ROOT, 'cleaned_data')
CLEANED_VIDEOS_FILE = os.path.join(CLEANED_DATA_DIR, 'filtered_videos.csv')
OUTPUT_IDS_FILE = os.path.join(CLEANED_DATA_DIR, 'ids_portuguese.txt')

df = pd.read_csv(CLEANED_VIDEOS_FILE)
df.head()

In [ ]:
df.shape

In [ ]:
def text_concat(row):
    titulo = str(row['title']) if pd.notna(row['title']) else ''
    descricao = str(row['description']) if pd.notna(row['description']) else ''
    
    full_text = f"{titulo} {descricao}"
    
    return full_text

---
## 1. Data Preparation

Loads the video dataset and prepares text for analysis:
- **Concatenation**: Combine title and description into a single text field
- **Normalization**: Convert text to lowercase
- **Tokenization**: Compute token counts (max 512 for the model)

In [ ]:
model_ckpt = 'papluca/xlm-roberta-base-language-detection'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
new_df = df[['video_id', 'title', 'description']].copy()
new_df['full_text'] = new_df.apply(text_concat, axis=1).str.lower()
new_df.shape

In [ ]:
new_df['num_tokens'] = new_df['full_text'].apply(lambda x: len(tokenizer.encode(x, truncation=True, max_length=512)))
new_df['num_tokens'].describe()

In [ ]:
new_df.head()

In [ ]:
BATCH_SIZE = 16

file_to_save = 'cleaned_data/ids_portuguese.txt'

portuguese_video_ids = []

all_full_texts = new_df['full_text'].tolist()
all_video_ids = new_df['video_id'].tolist()

---
## 2. Language Detection

Classifies the language of each video using **XLM-RoBERTa** (multilingual model):
- **Model**: papluca/xlm-roberta-base-language-detection (supports 20+ languages)
- **Processing**: Batch size 16 to optimize GPU usage
- **Pipeline**: Tokenization → Inference → Softmax → Classification
- **Filter**: Selects only videos classified as 'pt' (Portuguese)
- **Output**: Text file with Portuguese video IDs ({'ids_portuguese.txt'})

In [ ]:
for i in tqdm(range(0, len(all_full_texts), BATCH_SIZE), desc="Detecting languages"):
    batch_texts = all_full_texts[i : i + BATCH_SIZE]
    batch_video_ids = all_video_ids[i : i + BATCH_SIZE]

    inputs = tokenizer(batch_texts, truncation=True, padding=True, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    preds = torch.softmax(logits, dim=-1)

    id2lang = model.config.id2label
    predicted_lang_ids = torch.argmax(preds, dim=1).cpu().numpy()

    for j, predicted_lang_id in enumerate(predicted_lang_ids):
        video_id = batch_video_ids[j]
        predicted_lang = id2lang[predicted_lang_id]
        
        if predicted_lang == 'pt':
            portuguese_video_ids.append(str(video_id))

In [ ]:
with open(OUTPUT_IDS_FILE, 'w') as f:
    for video_id in portuguese_video_ids:
        f.write(f"{video_id}\n")

In [ ]:
print('Total videos in dataset:', len(df))
print('Total Portuguese videos in dataset:', len(portuguese_video_ids))
print('Proportion of Portuguese videos:', len(portuguese_video_ids) / len(df))
print('Saved file:', OUTPUT_IDS_FILE)